---
## BƯỚC 1: Cài đặt và import thư viện


In [1]:
# ==========================================
# TIKTOK CREATORS DATA AUDIT & SAMPLING FRAME
# Phase GĐ1: Audit & Sampling Design
# ==========================================

import os
import pymongo
import pandas as pd
import numpy as np
from datetime import datetime


---
## BƯỚC 2: Kết nối MongoDB


In [2]:
# Cấu hình kết nối
MONGO_URI = os.getenv("MONGO_URI", "mongodb://localhost:27017")  # Địa chỉ MongoDB
DB_NAME = "tiktok_creators_db"  # Tên database
COLLECTION_NAME = "creators_9k"  # Tên collection (bảng)

# Thực hiện kết nối
try:
    client = pymongo.MongoClient(MONGO_URI)
    db = client[DB_NAME]
    collection = db[COLLECTION_NAME]
    
    # Kiểm tra kết nối
    doc_count = collection.count_documents({})
    print(f"✓ Kết nối thành công!")
    print(f"✓ Database: {DB_NAME}")
    print(f"✓ Collection: {COLLECTION_NAME}")
    print(f"✓ Tổng số bản ghi: {doc_count:,}")
    
except Exception as e:
    print(f"✗ Lỗi kết nối: {e}")
    raise

✓ Kết nối thành công!
✓ Database: tiktok_creators_db
✓ Collection: creators_9k
✓ Tổng số bản ghi: 9,443


---
## BƯỚC 3: Tải dữ liệu từ MongoDB


In [3]:
# Chọn các trường (cột) cần lấy
fields_to_get = {
    '_id': 1,              # ID MongoDB
    'ID': 1,               # ID creator
    'Name': 1,             # Tên creator
    'Country': 1,          # Quốc gia
    'Followers': 1,        # Số followers
    'Engagement': 1,       # Tỷ lệ tương tác
    'Median Views': 1,     # Lượt xem trung vị
    'Start Price': 1,      # Giá khởi điểm
    'Broadcast Score': 1,  # Điểm phát sóng
    'Collab Score': 1,     # Điểm hợp tác
    'Tags': 1              # Danh mục
}

print("Đang tải dữ liệu từ MongoDB...")

# Lấy dữ liệu
data = list(collection.find({}, fields_to_get))

# Chuyển thành DataFrame
df = pd.DataFrame(data)

print(f"✓ Đã tải {len(df):,} bản ghi")
print(f"✓ Số cột: {len(df.columns)}")

# Đổi tên cột sang tiếng Anh dễ hiểu
df.rename(columns={
    'ID': 'creator_id',
    'Name': 'name',
    'Country': 'country',
    'Followers': 'followers',
    'Engagement': 'engagement',
    'Median Views': 'median_views',
    'Start Price': 'price',
    'Broadcast Score': 'broadcast_score',
    'Collab Score': 'collab_score',
    'Tags': 'category'
}, inplace=True)

# Xem 5 dòng đầu tiên
print("\n5 dòng dữ liệu đầu tiên:")
print(df.head())

Đang tải dữ liệu từ MongoDB...
✓ Đã tải 9,443 bản ghi
✓ Số cột: 11

5 dòng dữ liệu đầu tiên:
              _id broadcast_score collab_score   country engagement followers  \
0          caonho            98.6         79.2  Việt Nam     14,06%      4,5M   
1  maitrithuc2020            95.8         82.2  Việt Nam      9,15%      1,7M   
2         dntminh            92.3         77.2  Việt Nam     12,47%      2,6M   
3     hoangvinhhh            91.6         77.4  Việt Nam      6,60%       12M   
4       tuyenxutv            90.2         92.5  Việt Nam      8,91%      3,9M   

       creator_id median_views               name                price  \
0          caonho       372,6K            CÁO NHỎ  Thỏa thuận/Chưa đặt   
1  maitrithuc2020       691,9K       Mai Trí Thức  Thỏa thuận/Chưa đặt   
2         dntminh       588,8K        Vitamin Mèo  Thỏa thuận/Chưa đặt   
3     hoangvinhhh         1,3M  Nguyễn Hoàng Vinh       12.801.218 VND   
4       tuyenxutv          20K        Tuyền Xu TV 

In [4]:
df_head = df.head(100)  # Lấy 100 dòng đầu tiên để phân tích
df_head.to_excel("tiktok_creators_head_100.xlsx", index=False)  # Lưu ra file Excel

---
## Làm sạch và chuyển đổi dữ liệu

In [4]:
print("Đang làm sạch dữ liệu...\n")

# ===== Hàm chuyển đổi số followers =====
def convert_followers(value):
    """
    Chuyển '1,5M' -> 1500000
    Chuyển '500K' -> 500000
    """
    if pd.isna(value) or value == '':
        return None
    
    value = str(value).strip().replace(',', '.')
    
    if 'M' in value:
        return float(value.replace('M', '')) * 1_000_000
    elif 'K' in value:
        return float(value.replace('K', '')) * 1_000
    else:
        try:
            return float(value)
        except:
            return None

# ===== Hàm chuyển đổi tỷ lệ engagement =====
def convert_engagement(value):
    """
    Chuyển '14,06%' -> 14.06
    """
    if pd.isna(value) or value == '':
        return None
    
    value = str(value).strip().replace('%', '').replace(',', '.')
    try:
        return float(value)
    except:
        return None

# ===== Hàm chuyển đổi giá =====
def convert_price(value):
    """
    Chuyển '12.801.218 VND' -> 12801218
    'Thỏa thuận' -> None
    """
    if pd.isna(value) or value == '':
        return None
    
    value = str(value).strip()
    
    if 'Thỏa thuận' in value or 'Chưa đặt' in value:
        return None
    
    # Loại bỏ 'VND' và các ký tự không phải số
    value = value.replace('VND', '').replace('.', '').replace(',', '').strip()
    try:
        return float(value)
    except:
        return None

# ===== Áp dụng chuyển đổi =====
df['followers_num'] = df['followers'].apply(convert_followers)
df['engagement_num'] = df['engagement'].apply(convert_engagement)
df['median_views_num'] = df['median_views'].apply(convert_followers)
df['price_num'] = df['price'].apply(convert_price)

print("✓ Đã chuyển đổi followers")
print("✓ Đã chuyển đổi engagement")
print("✓ Đã chuyển đổi median_views")
print("✓ Đã chuyển đổi price")

# Xem kết quả
print("\nDữ liệu sau khi chuyển đổi:")
print(df[['name', 'followers', 'followers_num', 'price', 'price_num']].head())

Đang làm sạch dữ liệu...

✓ Đã chuyển đổi followers
✓ Đã chuyển đổi engagement
✓ Đã chuyển đổi median_views
✓ Đã chuyển đổi price

Dữ liệu sau khi chuyển đổi:
                name followers  followers_num                price   price_num
0            CÁO NHỎ      4,5M      4500000.0  Thỏa thuận/Chưa đặt         NaN
1       Mai Trí Thức      1,7M      1700000.0  Thỏa thuận/Chưa đặt         NaN
2        Vitamin Mèo      2,6M      2600000.0  Thỏa thuận/Chưa đặt         NaN
3  Nguyễn Hoàng Vinh       12M     12000000.0       12.801.218 VND  12801218.0
4        Tuyền Xu TV      3,9M      3900000.0       10.000.000 VND  10000000.0


##  Phân loại creators theo mức giá

In [5]:

# Tính ngưỡng phân vị (33% và 67%)
price_33 = df['price_num'].quantile(0.33)
price_67 = df['price_num'].quantile(0.67)

print(f"Ngưỡng 33%: {price_33:,.0f} VND")
print(f"Ngưỡng 67%: {price_67:,.0f} VND")
print()

# Hàm phân loại
def classify_price(price):
    if price < price_33:
        return 'low'
    elif price < price_67:
        return 'mid'
    else:
        return 'high'

# Áp dụng phân loại
df['price_tier'] = df['price_num'].apply(classify_price)

# Thống kê
price_stats = df['price_tier'].value_counts().sort_index()
print("Phân bổ theo mức giá:")
for tier, count in price_stats.items():
    percent = count / len(df) * 100
    print(f"  {tier.upper():8s}: {count:4,} creators ({percent:5.1f}%)")

Ngưỡng 33%: 1,000,000 VND
Ngưỡng 67%: 3,000,000 VND

Phân bổ theo mức giá:
  HIGH    : 7,226 creators ( 76.5%)
  LOW     :  920 creators (  9.7%)
  MID     : 1,297 creators ( 13.7%)


# Phân loại creators theo quy mô

In [9]:
print("Đang phân loại theo quy mô...\n")

# Định nghĩa ngưỡng
MICRO_THRESHOLD = 500_000   # 500K
MID_THRESHOLD = 2_000_000   # 2M
print(f"Ngưỡng MICRO: < {MICRO_THRESHOLD:,} followers")
print(f"Ngưỡng MID: {MICRO_THRESHOLD:,} - {MID_THRESHOLD:,} followers")
print(f"Ngưỡng MACRO: > {MID_THRESHOLD:,} followers")
# Hàm phân loại
def classify_size(followers):
    if followers < MICRO_THRESHOLD:
        return 'micro'
    elif followers < MID_THRESHOLD:
        return 'mid'
    else:
        return 'macro'

# Áp dụng phân loại
df['size_tier'] = df['followers_num'].apply(classify_size)

# Thống kê
size_stats = df['size_tier'].value_counts().sort_index()
print("Phân bổ theo quy mô:")
for tier, count in size_stats.items():
    percent = count / len(df) * 100
    print(f"  {tier.upper():8s}: {count:4,} creators ({percent:5.1f}%)")

Đang phân loại theo quy mô...

Ngưỡng MICRO: < 500,000 followers
Ngưỡng MID: 500,000 - 2,000,000 followers
Ngưỡng MACRO: > 2,000,000 followers
Phân bổ theo quy mô:
  MACRO   :  257 creators (  2.7%)
  MICRO   : 8,184 creators ( 86.7%)
  MID     : 1,002 creators ( 10.6%)


# Phân tích theo danh mục (Category)

In [7]:
# ===== DEMO: Hiểu cấu trúc dữ liệu category =====
print("Ví dụ dữ liệu category của 3 creators đầu tiên:\n")

for i in range(min(3, len(df))):
    name = df.iloc[i]['name']
    cats = df.iloc[i]['category']
    
    print(f"{i+1}. {name}")
    print(f"   Type: {type(cats)}")
    print(f"   Value: {cats}")
    
    # Nếu là string có dấu phẩy
    if isinstance(cats, str) and ',' in cats:
        cat_list = [c.strip() for c in cats.split(',')]
        print(f"   Số danh mục: {len(cat_list)}")
        for cat in cat_list[:3]:  # Chỉ hiển thị 3 danh mục đầu
            print(f"      - {cat}")
    print()


Ví dụ dữ liệu category của 3 creators đầu tiên:

1. CÁO NHỎ
   Type: <class 'str'>
   Value: Thương mại điện tử (không dùng ứng dụng), Hài kịch, Thực phẩm và Đồ uống, Tin tức và Giải trí
   Số danh mục: 4
      - Thương mại điện tử (không dùng ứng dụng)
      - Hài kịch
      - Thực phẩm và Đồ uống

2. Mai Trí Thức
   Type: <class 'str'>
   Value: Âm nhạc, Nhảy, Hát nhép
   Số danh mục: 3
      - Âm nhạc
      - Nhảy
      - Hát nhép

3. Vitamin Mèo
   Type: <class 'str'>
   Value: Thiết bị, Thú cưng, Sức khỏe, Chăm sóc thú cưng
   Số danh mục: 4
      - Thiết bị
      - Thú cưng
      - Sức khỏe



In [8]:
print("Phân tích theo danh mục nội dung...\n")

# ===== XỬ LÝ DANH MỤC DẠNG LIST =====

# Bước 1: Tách từng danh mục từ list
all_categories = []

for categories in df['category']:
    # Kiểm tra nếu là string (có dấu phẩy) thì split
    if isinstance(categories, str):
        cat_list = [c.strip() for c in categories.split(',')]
        all_categories.extend(cat_list)
    # Nếu là list thì thêm trực tiếp
    elif isinstance(categories, list):
        all_categories.extend(categories)

# Bước 2: Đếm tần suất từng danh mục
from collections import Counter
category_counter = Counter(all_categories)

# Bước 3: Lấy top 20 danh mục phổ biến
top_20 = category_counter.most_common(20)

print("Top 20 danh mục phổ biến nhất:")
print("(Lưu ý: 1 creator có thể có nhiều danh mục)\n")

total_tags = sum(category_counter.values())
for i, (cat, count) in enumerate(top_20, 1):
    percent = count / total_tags * 100
    print(f"{i:2d}. {count:4,} lượt ({percent:4.1f}%) - {cat}")

print(f"\nTổng số danh mục khác nhau: {len(category_counter):,}")
print(f"Tổng số lượt gắn tag: {total_tags:,}")
print(f"Trung bình mỗi creator có: {total_tags/len(df):.1f} danh mục")

Phân tích theo danh mục nội dung...

Top 20 danh mục phổ biến nhất:
(Lưu ý: 1 creator có thể có nhiều danh mục)

 1. 1,370 lượt ( 6.1%) - Hướng dẫn & mẹo làm đẹp
 2. 1,355 lượt ( 6.1%) - Cuộc sống hàng ngày
 3. 1,133 lượt ( 5.1%) - Hát nhép
 4. 1,062 lượt ( 4.8%) - Trang phục
 5.  936 lượt ( 4.2%) - Làm đẹp và chăm sóc cá nhân
 6.  827 lượt ( 3.7%) - Nhảy
 7.  821 lượt ( 3.7%) - Ảnh tự chụp
 8.  787 lượt ( 3.5%) - Tin tức và Giải trí
 9.  766 lượt ( 3.4%) - Trò chơi điện tử
10.  761 lượt ( 3.4%) - Âm nhạc
11.  624 lượt ( 2.8%) - Phim ảnh & Truyền hình
12.  542 lượt ( 2.4%) - Hài kịch
13.  532 lượt ( 2.4%) - Sản phẩm công nghệ & Thử nghiệm
14.  462 lượt ( 2.1%) - May mặc & Phụ kiện
15.  395 lượt ( 1.8%) - Hoạt hình & Cosplay
16.  380 lượt ( 1.7%) - Khám phá nhà hàng
17.  378 lượt ( 1.7%) - Sức khỏe
18.  356 lượt ( 1.6%) - Thực phẩm và Đồ uống
19.  339 lượt ( 1.5%) - Mukbang & Nếm thức ăn
20.  337 lượt ( 1.5%) - Nấu ăn & Công thức nấu ăn

Tổng số danh mục khác nhau: 103
Tổng số lượt gắn 

Nhóm nội dung chiếm ưu thế:

- (1+4+5+14)    Beauty & Fashion : ~3,768 lượt (~17%)
- (3+6+10+12)   Entertainment    : ~3,252 lượt (~15%)
- (2+16+19+20)  Lifestyle        : ~2,409 lượt (~11%)

# Trực quan sâu về những kênh có quy mô cao